
# Laboratory 4B — q-point convergence of harmonic thermodynamics

We now sample the first Brillouin zone progressively:

```text
1×1×1  →  2×2×2  →  3×3×3  →  4×4×4
```

You calculate the $2\times2\times2$ result. The denser calculations are supplied
as instructor reference data.



## 1. Prepare the 2×2×2 phonon calculation

Open:

```text
inputs/mgo_2x2x2_template.d12
```

As before, replace `<A0>` with the optimized Laboratory 3 lattice parameter and
`<X>` with the SHRINK value selected in Laboratory 2.

The new part is:

```text
SCELPHONO
2 0 0
0 2 0
0 0 2
FREQCALC
DISPERSI
NOOPTGEOM
NOINTENS
ENDFREQ
```

The supercell is a direct-space construction. `DISPERSI` asks CRYSTAL to
evaluate the commensurate phonon q mesh associated with that supercell.



### Question 1

How many points are present in the **full** reciprocal-space mesh associated
with a $2\times2\times2$ isotropic supercell?

**Your answer:**  
_Write your answer here._



## 2. Run the 2×2×2 calculation

Run:

```bash
cluminer-crystal mgo_2x2x2.d12 --cores-per-node 2 --threads 2
```

and place the completed output in:

```text
outputs/mgo_2x2x2.out
```


In [ ]:

from pathlib import Path

from mineral_science_tools.phonons import (
    convergence_table,
    entropy_convergence_table,
    create_ha_yaml,
    full_qpoint_table,
    ha_cv_curve,
    ha_entropy_curve,
    irreducible_qpoint_table,
    load_experimental_heat_capacity,
    load_experimental_entropy,
    phonon_input_overview,
    plot_cv_comparison,
    plot_entropy_comparison,
    prefer_student_output,
    write_irreducible_phonon_yaml,
)

OUT_GAMMA = prefer_student_output(
    "outputs/mgo_gamma.out",
    "reference-results/gamma/mgo_gamma.out",
)
OUT_2 = prefer_student_output(
    "outputs/mgo_2x2x2.out",
    "reference-results/2x2x2/mgo_2x2x2.out",
)
OUT_3 = Path("reference-results/3x3x3/mgo_3x3x3.out")
OUT_4 = Path("reference-results/4x4x4/mgo_4x4x4.out")

OUTPUTS = {
    "Gamma": OUT_GAMMA,
    "2x2x2": OUT_2,
    "3x3x3": OUT_3,
    "4x4x4": OUT_4,
}

for label, path in OUTPUTS.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {label} output: {path}")

OUTPUTS



## 3. Normalize all four calculations with Quantas


In [ ]:

YAMLS = {}

for label, output in OUTPUTS.items():
    safe = label.lower().replace("x", "x").replace("gamma", "gamma")
    destination = Path(f"mgo_{safe}.yaml")
    create_ha_yaml(
        output,
        destination,
        jobname=f"MgO {label} phonon sampling",
        formula_units=1,
    )
    YAMLS[label] = destination

overview = {
    label: phonon_input_overview(path)
    for label, path in YAMLS.items()
}
overview



The full CRYSTAL datasets contain 1, 8, 27 and 64 q-point spectra.

Quantas can use those full uniform meshes directly with equal weights.



## 4. Group symmetry-equivalent q-points

For interpretation, we can compress the full mesh into irreducible
representatives with multiplicity weights.

The grouping is determined from the crystal symmetry with spglib. Frequencies
are checked only *afterwards* to verify that symmetry-related q-points have
consistent spectra.


In [ ]:

ir_2 = irreducible_qpoint_table(YAMLS["2x2x2"])
ir_2



### Question 2

How many irreducible q-points remain after symmetry reduction of the
$2\times2\times2$ mesh?

What is the sum of their multiplicities, and why must it reproduce the number
of points in the full mesh?

**Your answer:**  
_Write 2–3 lines here._



Optionally, write a reduced Quantas YAML and verify that the q-point weights
carry exactly the same integration information as the full mesh:


In [ ]:

reduced_2 = write_irreducible_phonon_yaml(
    YAMLS["2x2x2"],
    "mgo_2x2x2_irreducible.yaml",
)

phonon_input_overview(reduced_2)



The thermodynamic calculation below will continue to use the original full
meshes. The reduced file is included to show explicitly what a
Phonopy-like irreducible-q representation means.


## 5. Calculate harmonic entropy for all q meshes

Before examining heat capacity, compare the entropy.

For a macroscopic crystal, the entropy depends on the thermally accessible
phonon states throughout the Brillouin zone. The progression from $\Gamma$ to
denser q meshes should therefore produce a converging sequence of
$S(T)$ curves.

The experimental/reference points plotted below are standard molar entropies
at approximately 1 bar from the NIST-JANAF thermochemical evaluation. The
Quantas curves are harmonic entropies calculated at the fixed DFT equilibrium
volume.


In [ ]:
ENTROPY_CURVES = {
    label: ha_entropy_curve(
        yaml_path,
        temperature_min=0.0,
        temperature_max=1200.0,
        temperature_step=10.0,
    )
    for label, yaml_path in YAMLS.items()
}

experimental_entropy = load_experimental_entropy(
    "data/mgo_experimental_entropy.csv"
)


In [ ]:
plot_entropy_comparison(
    ENTROPY_CURVES,
    experimental_entropy,
);


### Question 3

Describe the convergence of $S(T)$ from $\Gamma$ to $2\times2\times2$,
$3\times3\times3$ and $4\times4\times4$.

Does the entropy comparison with the experimental/reference data look better
than the $\Gamma$-only result?

**Your answer:**  
_Write 3–4 lines here._


In [ ]:
entropy_convergence_table(
    ENTROPY_CURVES,
    temperatures=(300.0, 600.0, 1000.0),
    reference_label="4x4x4",
)


### Question 4

Use the table to decide whether the entropy is numerically converged with
respect to q-point sampling.

After q-point convergence, why can a small systematic difference from the
standard-pressure experimental/reference entropy still remain?

**Your answer:**  
_Write 3–4 lines here._



## 6. Calculate harmonic $C_V$ for all q meshes


In [ ]:

CURVES = {
    label: ha_cv_curve(
        yaml_path,
        temperature_min=0.0,
        temperature_max=1200.0,
        temperature_step=10.0,
    )
    for label, yaml_path in YAMLS.items()
}

experimental = load_experimental_heat_capacity(
    "data/mgo_experimental_heat_capacity.csv"
)


In [ ]:

plot_cv_comparison(CURVES, experimental);



### Question 5

Describe how the calculated heat-capacity curve changes from $\Gamma$ to
$2\times2\times2$, then to $3\times3\times3$ and $4\times4\times4$.

At which temperatures is q-point sampling most visibly important?

**Your answer:**  
_Write 4–5 lines here._



## 7. Quantify $C_V$ convergence

Use the $4\times4\times4$ result as the numerical reference for this exercise.


In [ ]:

convergence_table(
    CURVES,
    temperatures=(100.0, 300.0, 1000.0),
    reference_label="4x4x4",
)



### Question 6

At 100, 300 and 1000 K, compare the differences relative to the
$4\times4\times4$ result.

Would you consider the $3\times3\times3$ mesh converged for the interpretation
performed here? Justify your answer from the table rather than from the mesh
size alone.

**Your answer:**  
_Write 3–5 lines here._



## 8. The high-temperature limit

For MgO, the harmonic Dulong–Petit limit is

$$
3nR = 6R \approx 49.9\ \mathrm{J\,mol^{-1}\,K^{-1}}.
$$

It is drawn as a dashed horizontal line in the comparison plot.

### Question 7

Does the well-sampled harmonic $C_V$ approach the expected high-temperature
limit?

How does the experimental heat capacity behave relative to that limit at high
temperature?

**Your answer:**  
_Write 3–4 lines here._



## 9. Numerical convergence does not remove model error

By the time the $3\times3\times3$ and $4\times4\times4$ curves are close, the
Brillouin-zone integration is no longer the dominant explanation for a
remaining difference from experiment.

Remember:

```text
calculation: harmonic Cv at fixed volume
experiment:  approximately Cp at ambient pressure
```

### Question 8

Why does the harmonic approximation predict no thermal expansion?

What consequence does this have for the distinction between $C_P$ and $C_V$?

**Your answer:**  
_Write 3–4 lines here._



### Question 9 — Final interpretation

The $4\times4\times4$ result is numerically well sampled but still does not
reproduce every experimental value exactly.

List the main approximations or physical effects that can explain the remaining
difference.

Separate explicitly:

- numerical q-point convergence;
- harmonic/fixed-volume limitations;
- $C_V$ versus experimental $C_P$;
- other possible model errors.

**Your answer:**  
_Write 5–7 lines here._



## 10. What would a possible Laboratory 5 change?

A quasi-harmonic calculation repeats harmonic phonon calculations at several
volumes. The frequencies then depend on volume:

$$
\nu_{\mathbf qj} = \nu_{\mathbf qj}(V).
$$

This allows the equilibrium volume to change with temperature and introduces
thermal expansion and a non-zero $C_P-C_V$ correction.

That extension is optional and is **not** required for this laboratory.



## Before you finish

Save the notebooks and compact processed files, then:

```text
save → commit → Sync Changes → verify on GitHub → stop Codespace
```
